In [12]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torch.optim as optim

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [14]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

In [15]:
train_datasets = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_datasets = datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

c:\Users\oguzh\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [16]:
train_datasets, val_datasets = random_split(train_datasets, lengths= [0.85,0.15])

len(train_datasets), len(val_datasets)

(42500, 7500)

In [17]:
train_dataloader = DataLoader(
    dataset= train_datasets,
    batch_size= 32,
    shuffle = True,
    num_workers= 4,
    pin_memory= True,
)

val_dataloader = DataLoader(
    dataset= val_datasets,
    batch_size= 32,
    shuffle= False,
    num_workers= 4,
    pin_memory= True
)

test_dataloader = DataLoader(
    dataset= test_datasets,
    batch_size= 32,
    shuffle = False,
    num_workers= 4,
    pin_memory= True
)

In [18]:
class InvertedResidual(nn.Module):
    def __init__(self, in_channels, out_channels, stride, expansion):
        super().__init__()

        hidden = in_channels * expansion

        self.use_residual = (
            stride == 1 and
            in_channels == out_channels
        )

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                hidden,
                kernel_size=1,
                bias=False
            ),
            nn.BatchNorm2d(hidden),
            nn.ReLU6(inplace=True),

            nn.Conv2d(
                hidden,
                hidden,
                kernel_size=3,
                stride=stride,
                padding=1,
                groups=hidden,
                bias=False
            ),
            nn.BatchNorm2d(hidden),
            nn.ReLU6(inplace=True),

            nn.Conv2d(
                hidden,
                out_channels,
                kernel_size=1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x):

        if self.use_residual:
            return x + self.block(x)

        return self.block(x)

In [24]:
class MobileNetV2(nn.Module):
    def __init__(self, num_classes=1000):
        super().__init__()

        self.cfgs = [ # Expansion, out_channels, num_blocks, stride
            [1,  16, 1, 1],
            [6,  24, 2, 2],
            [6,  32, 3, 2],
            [6,  64, 4, 2],
            [6,  96, 3, 1],
            [6, 160, 3, 2],
            [6, 320, 1, 1],
        ]

        in_channels = 32

        layer = [
            nn.Conv2d(in_channels=3, out_channels=in_channels, kernel_size=3, padding=1, stride=2, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU6(inplace=True)
        ]

        for t, c, n, s in self.cfgs:
            for i in range(n):
                stride = s if i == 0 else 1

                layer.append(
                    InvertedResidual(in_channels=in_channels, out_channels=c, stride=stride, expansion=t)
                )
                in_channels = c

        out_channels = 1280
        
        layer.extend([
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU6(inplace=True)
        ])

        self.features = nn.Sequential(*layer)
        
        self.mean = nn.AdaptiveAvgPool2d(1) 
        
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2), 
            nn.Linear(in_features=out_channels, out_features=num_classes)
        )

    def forward(self, x):
        out = self.features(x)
        out = self.mean(out)
        out = torch.flatten(out, 1)     
        out = self.classifier(out)
        return out

num_classes = len(train_datasets.dataset.classes)
MobileNetV2_model = MobileNetV2(num_classes = num_classes).to(device)

In [ ]:
# Train && Validation Loop
epochs = 10

for epoch in range(epochs):
    MobileNetV2_model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for (X, y) in train_dataloader:
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        
        y_pred = MobileNetV2_model(X)
        loss = criterion(y_pred, y)
        
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X.size(0)
        _, predicted = torch.max(y_pred, 1)
        train_total += y.size(0)
        train_correct += (predicted == y).sum().item()

    train_epoch_loss = train_loss / train_total
    train_epoch_acc = train_correct / train_total

    MobileNetV2_model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.inference_mode():
        for (X, y) in val_dataloader:
            X = X.to(device)
            y = y.to(device)

            y_pred = MobileNetV2_model(X)
            loss = criterion(y_pred, y)
            
            val_loss += loss.item() * X.size(0)
            _, predicted = torch.max(y_pred, 1)
            val_total += y.size(0)
            val_correct += (predicted == y).sum().item()

    val_epoch_loss = val_loss / val_total
    val_epoch_acc = val_correct / val_total

    print(f"Epoch [{epoch+1}/{epochs}] | "
          f"Train Loss: {train_epoch_loss:.4f} - Train Acc: {train_epoch_acc:.4f} | "
          f"Val Loss: {val_epoch_loss:.4f} - Val Acc: {val_epoch_acc:.4f}")